In [2]:
import numpy as np

sites = [
    "GAGGTAAAC",
    "TCCGTAAGC",
    "CAGGTTGGA",
    "ACAGTCAGC",
    "TAGGTCAGC",
    "CAGGTCAGC",
    "CAGGTCGAT",
    "CAGGTCAGC",
    "CAGGTCAGC",
    "CAGGTTGGC"]

NUCLEOTIDES = ["A", "C", "G", "T"]
BG = {"A": 0.295, "C": 0.205, "G": 0.205, "T": 0.295}


# PFM
def build_pfm(sites: list[str]) -> np.ndarray:
    L = len(sites[0])
    pfm = np.zeros((4, L), dtype=int)
    for site in sites:
        for pos, nuc in enumerate(site):
            row = NUCLEOTIDES.index(nuc)
            pfm[row, pos] += 1
    return pfm


# PPM 
def pfm_to_ppm(pfm: np.ndarray, alpha: float = 0.1) -> np.ndarray:
    
    N = pfm.sum(axis=0)[0]         
    ppm = (pfm + alpha) / (N + 4 * alpha)
    return ppm


# PWM
def ppm_to_pwm(ppm: np.ndarray, bg: dict = BG) -> np.ndarray:

    bg_vec = np.array([BG[n] for n in NUCLEOTIDES]).reshape(-1, 1)
    pwm = np.log2(ppm / bg_vec)
    return pwm


# min / max
def pwm_score_range(pwm: np.ndarray):
    
    max_indices = np.argmax(pwm, axis=0)
    min_indices = np.argmin(pwm, axis=0)

    max_score = pwm[max_indices, np.arange(pwm.shape[1])].sum()
    min_score = pwm[min_indices, np.arange(pwm.shape[1])].sum()

    max_seq = "".join(NUCLEOTIDES[i] for i in max_indices)
    min_seq = "".join(NUCLEOTIDES[i] for i in min_indices)

    return max_score, max_seq, min_score, min_seq


def print_matrix(name: str, matrix: np.ndarray, fmt: str = "{:8.4f}"):
    L = matrix.shape[1]
    print(f"  {name} ")
    header = "     " + "".join(f"  pos{j+1}" for j in range(L))
    print(header)
    for i, nuc in enumerate(NUCLEOTIDES):
        row_str = f"  {nuc}  " + "".join(fmt.format(matrix[i, j]) for j in range(L))
        print(row_str)


if __name__ == "__main__":
    pfm = build_pfm(sites)
    ppm = pfm_to_ppm(pfm, alpha=0.1)
    pwm = ppm_to_pwm(ppm)

    print_matrix("PFM", pfm, fmt="{:7d}")
    print_matrix("PPM", ppm)
    print_matrix("PWM", pwm)

    max_score, max_seq, min_score, min_seq = pwm_score_range(pwm)

    print("  Теоретические экстремумы скора")
    print(f"  MAX скор : {max_score:+.4f}   последовательность: {max_seq}")
    print(f"  MIN скор : {min_score:+.4f}   последовательность: {min_seq}")

  PFM 
       pos1  pos2  pos3  pos4  pos5  pos6  pos7  pos8  pos9
  A        1      8      1      0      0      2      7      2      1
  C        6      2      1      0      0      6      0      0      8
  G        1      0      8     10      0      0      3      8      0
  T        2      0      0      0     10      2      0      0      1
  PPM 
       pos1  pos2  pos3  pos4  pos5  pos6  pos7  pos8  pos9
  A    0.1058  0.7788  0.1058  0.0096  0.0096  0.2019  0.6827  0.2019  0.1058
  C    0.5865  0.2019  0.1058  0.0096  0.0096  0.5865  0.0096  0.0096  0.7788
  G    0.1058  0.0096  0.7788  0.9712  0.0096  0.0096  0.2981  0.7788  0.0096
  T    0.2019  0.0096  0.0096  0.0096  0.9712  0.2019  0.0096  0.0096  0.1058
  PWM 
       pos1  pos2  pos3  pos4  pos5  pos6  pos7  pos8  pos9
  A   -1.4798  1.4006 -1.4798 -4.9392 -4.9392 -0.5469  1.2105 -0.5469 -1.4798
  C    1.5166 -0.0218 -0.9547 -4.4141 -4.4141  1.5166 -4.4141 -4.4141  1.9257
  G   -0.9547 -4.4141  1.9257  2.2441 -4.4141 -4.4141  